# 01c. QuartzNet + CR-CTC — регуляризация консистентностью

## Идея

CR-CTC (Consistency Regularization CTC) вводит дополнительный loss на симметричное KL-расстояние
между предсказаниями двух аугментированных версий одного и того же аудио.
Это побуждает энкодер давать устойчивые предсказания при вариациях входа.

**Два view одного батча.** В идеале оба view генерируются в DataLoader.
Однако в Wave-1 параметр `return_two_views` удалён из `SpokenNumbersDataset`.

**Упрощение (для демонстрации):** второй view генерируется на лету в цикле:
батч уже содержит augmented audio (view 1 через Dataset), а view 2 — результат
применения второго аугментера к тому же сырому аудио. Для этого в тренировочном цикле
audio передаётся через `aug2` напрямую (поверх уже аугментированного Dataset-ом).

> Упрощение: второй view делается на лету из уже-аугментированного Dataset-ом audio,
> что не идеально (view2 — это «aug поверх aug», а не независимый вид оригинала).
> В production рекомендуется dataset с двумя view'ами.

**Адаптация под Wave-1 Batch:** поля `audio_view2` / `audio_view2_lengths` удалены из `Batch`.
View 2 строится в цикле через отдельный `AudioAugmenter`.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути

`DATA_ROOT`, `CKPT_ROOT` и сопутствующие пути автоматически резолвятся в ячейке ниже относительно расположения ноутбука (`notebooks/experiments/..`). Правки требуются, только если структура репозитория изменена.

In [1]:
# Idempotent data download: populates ../data/ with train/ dev/ test/ and
# their CSVs from the shared Google Drive ZIP. No-op if already present.
from pathlib import Path
import zipfile

NOTEBOOK_DIR = Path.cwd()  # notebooks/experiments/
DATA_ROOT = (NOTEBOOK_DIR / ".." / "data").resolve()
ZIP_PATH = (NOTEBOOK_DIR / ".." / "data.zip").resolve()

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        import gdown
        gdown.download(
            url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
            output=str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT.parent)
    print(f"Extracted to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT}")


Data already present at /home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data


In [2]:
# Env hardening — MUST run before `import torch` in this process.
# PYTORCH_CUDA_ALLOC_CONF reduces VRAM fragmentation on torch.compile +
# variable T batches; cudnn.benchmark=False avoids autotune thrash under
# padded, variable-length inputs; matmul_precision="high" enables TF32.
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import torch

torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8


In [5]:
# Paths — auto-resolved from DATA_ROOT defined in the download cell.
from pathlib import Path
import torch

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("../../checkpoints") / "01c_cr_ctc"

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

MUSAN_ROOT = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT   = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"

assert MUSAN_ROOT.exists(), f"musan not found: {MUSAN_ROOT}"
assert RIR_ROOT.exists(),   f"RIRs not found:  {RIR_ROOT}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

device=cuda, train=/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/train, dev=/home/dench/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks/data/data/dev, ckpts=../../checkpoints/01c_cr_ctc


In [6]:
import gc
import json
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best, load_checkpoint
from gp1.train.metrics import compute_cer, compute_cer_in_out_harmonic, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig


In [7]:
# ---------------------------------------------------------------------------
# CRCTCLoss — скопировано из src/gp1/losses/cr_ctc.py (удалён в Wave 1)
# ---------------------------------------------------------------------------

class CRCTCLoss(nn.Module):
    """Consistency Regularization CTC Loss (CR-CTC).

    Computes symmetric KL divergence between two augmented views of the
    same utterance. Encourages the encoder to produce consistent predictions
    under different augmentations.

    Args:
        temperature: Softmax temperature for sharpening / smoothing (default 1.0).
        min_prob: Minimum confidence threshold; only frames where at least one
            view has max_prob >= min_prob contribute to the loss (default 0.1).
    """

    def __init__(self, temperature: float = 1.0, min_prob: float = 0.1) -> None:
        super().__init__()
        self.temperature = temperature
        self.min_prob = min_prob

    def forward(
        self,
        log_probs_a: torch.Tensor,     # [B, T, V] — view 1
        log_probs_b: torch.Tensor,     # [B, T, V] — view 2
        input_lengths: torch.Tensor,   # [B]
    ) -> torch.Tensor:
        assert log_probs_a.dim() == 3, "log_probs must be [B, T, V]"
        batch, time, _ = log_probs_a.shape

        if self.temperature != 1.0:
            log_probs_a = F.log_softmax(log_probs_a / self.temperature, dim=-1)
            log_probs_b = F.log_softmax(log_probs_b / self.temperature, dim=-1)

        probs_a = log_probs_a.exp()
        probs_b = log_probs_b.exp()

        lengths_mask = (
            torch.arange(time, device=log_probs_a.device)
            .unsqueeze(0)
            .lt(input_lengths.unsqueeze(1))
        )

        if self.min_prob > 0.0:
            conf_a = probs_a.max(dim=-1).values >= self.min_prob
            conf_b = probs_b.max(dim=-1).values >= self.min_prob
            conf_mask = conf_a | conf_b
        else:
            conf_mask = torch.ones(batch, time, dtype=torch.bool, device=log_probs_a.device)

        mask = lengths_mask & conf_mask
        n_valid = mask.sum().clamp(min=1)

        # Symmetric KL: 0.5 * (KL(a||b) + KL(b||a))
        kl_ab = (probs_a * (log_probs_a - log_probs_b)).sum(dim=-1)
        kl_ba = (probs_b * (log_probs_b - log_probs_a)).sum(dim=-1)
        sym_kl = 0.5 * (kl_ab + kl_ba)

        loss = (sym_kl * mask.float()).sum() / n_valid
        return loss

In [8]:
# --- Manifests + vocab ---
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
vocab = CharVocab()
print(f"Train: {len(train_records)} records. Dev: {len(dev_records)} records.")
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

# In-domain / out-of-domain speaker split for harmonic CER metric.
in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(
    f"dev speakers: in-domain={in_domain_count} samples, "
    f"out-of-domain={out_of_domain_count} samples"
)


Train: 12553 records. Dev: 2265 records.
Vocab size: 35, blank_id: 0
dev speakers: in-domain=600 samples, out-of-domain=1665 samples


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [9]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 30,
    "grad_accum": 2,
    "subsample_factor": 2,
}
print("FIXED:", FIXED)


FIXED: {'samplerate': 16000, 'n_fft': 512, 'n_mels': 80, 'hop_length': 160, 'win_length': 400, 'max_epochs': 30, 'grad_accum': 2, 'subsample_factor': 2}


In [10]:
HP_GRID = {
    "d_model": [192, 256],
    "dropout": [0.05, 0.10, 0.15],
    "lr": [0.005, 0.010, 0.015],
    "weight_decay": [1e-4, 1e-3],
    "warmup_steps": [1000, 2000, 3000],
    "min_lr_ratio": [0.01, 0.05],
    "specaug_freq_mask_param": [10, 15, 20],
    "specaug_num_freq_masks": [1, 2],
    "specaug_time_mask_param": [20, 25, 35],
    "specaug_num_time_masks": [2, 5],
    "specaug_time_mask_max_ratio": [0.04, 0.05, 0.08],
    "speed_prob": [0.5, 1.0],
    "speed_factors": [(0.9, 1.0, 1.1), (0.85, 1.0, 1.15)],
    "pitch_prob": [0.0, 0.3],
    "pitch_range_semitones": [(-2.0, 2.0), (-3.0, 3.0)],
    "gain_prob": [0.3, 0.7],
    "gain_db_range": [(-4.0, 4.0), (-8.0, 8.0)],
    "vtlp_prob": [0.0, 0.3],
    "vtlp_alpha_range": [(0.9, 1.1), (0.85, 1.15)],
    "noise_prob": [0.0, 0.3],
    "noise_snr_db_range": [(10.0, 20.0), (5.0, 20.0)],
    "rir_prob": [0.0, 0.3],
    "cr_ctc_weight": [0.05, 0.1, 0.2],
    "cr_min_prob": [0.05, 0.1],
}

N_TRIALS = 5
SEED = 42
random.seed(SEED)

print("N_TRIALS:", N_TRIALS)


N_TRIALS: 5


In [11]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)
# Move tensors to shared memory so DataLoader worker processes reuse the
# same underlying buffers instead of copying on every fork.
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()
print(f"AUDIO_CACHE: {len(AUDIO_CACHE)} entries")


preload audio: 100%|██████████| 14818/14818 [00:13<00:00, 1064.94it/s]


AUDIO_CACHE: 14818 entries


In [12]:
def _worker_init(worker_id: int) -> None:
    """1 BLAS-thread per worker + per-worker RNG seed for augmenter."""
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


In [13]:
# DataLoader / augmenter / SpecAug — пересоздаются на каждом trial внутри Шага 5.
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")
print(f"AUDIO_CACHE entries: {len(AUDIO_CACHE)}")


Train records: 12553, Dev records: 2265
AUDIO_CACHE entries: 14818


## Шаг 5. HP Random Search — Training Loop

Каждый trial семплирует значения из `HP_GRID`, пересоздаёт hp-зависимые объекты
(включая два `SpecAugmenter` с разными seeds и `CRCTCLoss(min_prob=hp["cr_min_prob"])`),
запускает развёрнутый custom training loop с two-view forward и сохраняет best
checkpoint в `trial_dir`.


In [14]:
mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)


In [ ]:
BATCH_SIZE = 64
DL_WORKERS = 4

trial_results = []
run_root = CKPT_ROOT / f"01c_cr_ctc_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    trial_dir = run_root / f"trial_{trial:02d}"
    trial_dir.mkdir(parents=True, exist_ok=True)

    aug_cfg = AugConfig(
        speed_factors=tuple(hp["speed_factors"]),
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        pitch_range_semitones=tuple(hp["pitch_range_semitones"]),
        gain_prob=hp["gain_prob"],
        gain_db_range=tuple(hp["gain_db_range"]),
        seed=SEED + trial,
    )
    augmenter1 = AudioAugmenter(aug_cfg)
    gpu_aug = GPUAudioAugmenter(
        samplerate=FIXED["samplerate"],
        vtlp_prob=hp["vtlp_prob"],
        vtlp_alpha_range=tuple(hp["vtlp_alpha_range"]),
        noise_prob=hp["noise_prob"],
        noise_snr_db_range=tuple(hp["noise_snr_db_range"]),
        musan_root=MUSAN_ROOT,
        rir_prob=hp["rir_prob"],
        rir_root=RIR_ROOT,
    ).to(device)

    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial,
    )
    spec_aug2 = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial + 10000,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        target_samplerate=FIXED["samplerate"],
        augmenter=augmenter1,
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        target_samplerate=FIXED["samplerate"],
        augmenter=None,
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=DL_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=DL_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )

    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)
    model = torch.compile(model, mode="default", dynamic=True)

    cr_loss_fn = CRCTCLoss(temperature=1.0, min_prob=hp["cr_min_prob"])
    ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)
    all_params = list(model.parameters())
    optimizer = build_novograd(
        all_params,
        lr=hp["lr"],
        betas=(0.95, 0.5),
        weight_decay=hp["weight_decay"],
    )
    steps_per_epoch = math.ceil(len(train_loader) / FIXED["grad_accum"])
    total_steps = steps_per_epoch * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=hp["min_lr_ratio"],
    )

    best_cer = float("inf")
    best_ckpt_path = None

    for epoch in tqdm(range(FIXED["max_epochs"]), desc=f"trial {trial} epochs"):
        model.train()
        spec_aug.train()
        spec_aug2.train()
        optimizer.zero_grad()
        grad_step = 0

        for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
            audio_raw = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            targets = batch.targets.to(device)
            target_lengths = batch.target_lengths.to(device)

            if model.training:
                audio_v1 = gpu_aug(audio_raw, audio_lengths)
                audio_v2 = gpu_aug(audio_raw, audio_lengths)
            else:
                audio_v1 = audio_v2 = audio_raw

            with torch.no_grad():
                mel_v1 = mel_extractor(audio_v1)

            mel_lengths = (
                (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
            ).clamp(max=mel_v1.size(-1))

            mel_v1 = spec_aug(mel_v1, mel_lengths)

            with torch.no_grad():
                mel_v2_base = mel_extractor(audio_v2)
            assert mel_v2_base.size(-1) == mel_v1.size(-1), (
                f"GPU aug must preserve T: v1={mel_v1.size(-1)}, v2={mel_v2_base.size(-1)}"
            )
            mel_v2 = spec_aug2(mel_v2_base, mel_lengths)

            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                 enabled=(device.type == "cuda")):
                out_v1 = model(mel_v1, mel_lengths)
                out_v2 = model(mel_v2, mel_lengths)

            with torch.autocast(device_type=device.type, enabled=False):
                loss_main = ctc_loss_fn(
                    out_v1.log_probs.float(), targets, out_v1.output_lengths, target_lengths
                )
                loss_cr = cr_loss_fn(
                    out_v1.log_probs.float(), out_v2.log_probs.float(), out_v1.output_lengths
                )

            loss = loss_main + hp["cr_ctc_weight"] * loss_cr

            (loss / FIXED["grad_accum"]).backward()
            grad_step += 1
            if grad_step % FIXED["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        model.eval()
        spec_aug.eval()
        spec_aug2.eval()
        all_refs, all_hyps, all_spks = [], [], []

        with torch.no_grad():
            for batch in dev_loader:
                audio = batch.audio.to(device)
                audio_lengths = batch.audio_lengths.to(device)
                mel = mel_extractor(audio)
                mel_lengths = (
                    (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
                ).clamp(max=mel.size(-1))
                out = model(mel, mel_lengths)
                hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
                all_refs.extend(batch.transcriptions)
                all_hyps.extend(hyps)
                all_spks.extend(batch.spk_ids)

        ref_words = [digits_to_words(r) for r in all_refs]
        val_cer = compute_cer(ref_words, all_hyps)
        in_cer, out_cer, hm_cer = compute_cer_in_out_harmonic(
            ref_words, all_hyps, all_spks, in_domain_speakers
        )

        tqdm.write(
            f"trial {trial} epoch {epoch:3d} | hm_cer={hm_cer:.4f} "
            f"(in={in_cer:.4f} out={out_cer:.4f}) | val_cer={val_cer:.4f}"
        )

        if hm_cer < best_cer:
            best_cer = hm_cer
            best_ckpt_path = save_best(
                model,
                meta=dict(
                    arch="quartznet10x4_cr_ctc",
                    hparams={**FIXED, **hp},
                    hm_cer=hm_cer,
                    val_cer=val_cer,
                    in_cer=in_cer,
                    out_cer=out_cer,
                    epoch=epoch,
                    trial=trial,
                ),
                ckpt_dir=trial_dir,
            )
            tqdm.write(f"  Checkpoint saved: {best_ckpt_path}")

    trial_results.append({
        "trial": trial,
        "hp": hp,
        "best_monitored": best_cer,
        "best_ckpt_path": best_ckpt_path,
    })

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    del model, optimizer, scheduler, ctc_loss_fn, cr_loss_fn
    del train_loader, dev_loader, train_ds, dev_ds
    del augmenter1, gpu_aug, spec_aug, spec_aug2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    torch._dynamo.reset()

print("\nHP search complete.")



=== Trial 1/5 ===
{
  "samplerate": 16000,
  "n_fft": 512,
  "n_mels": 80,
  "hop_length": 160,
  "win_length": 400,
  "max_epochs": 30,
  "grad_accum": 2,
  "subsample_factor": 2,
  "d_model": 192,
  "dropout": 0.05,
  "lr": 0.015,
  "weight_decay": 0.001,
  "warmup_steps": 1000,
  "min_lr_ratio": 0.01,
  "specaug_freq_mask_param": 10,
  "specaug_num_freq_masks": 1,
  "specaug_time_mask_param": 35,
  "specaug_num_time_masks": 2,
  "specaug_time_mask_max_ratio": 0.08,
  "speed_prob": 1.0,
  "speed_factors": [
    0.9,
    1.0,
    1.1
  ],
  "pitch_prob": 0.0,
  "pitch_range_semitones": [
    -2.0,
    2.0
  ],
  "gain_prob": 0.3,
  "gain_db_range": [
    -4.0,
    4.0
  ],
  "vtlp_prob": 0.0,
  "vtlp_alpha_range": [
    0.9,
    1.1
  ],
  "noise_prob": 0.3,
  "noise_snr_db_range": [
    10.0,
    20.0
  ],
  "rir_prob": 0.3,
  "cr_ctc_weight": 0.2,
  "cr_min_prob": 0.1
}


trial 0 epochs:   0%|          | 0/30 [00:00<?, ?it/s]W0425 17:15:32.433000 88241 torch/_inductor/utils.py:1613] [0/0_1] Not enough SMs to use max_autotune_gemm mode


## Шаг 6. Сводный отчёт трайлов


In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_monitored"])
print(f"{'trial':>5} {'best_cer':>10} {'lr':>8} {'dropout':>8} {'d_model':>8} {'cr_w':>6} {'cr_minp':>7}")
print("-" * 60)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_monitored']:>10.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
        f" {hp['cr_ctc_weight']:>6.2f}"
        f" {hp['cr_min_prob']:>7.2f}"
    )


## Шаг 7. Валидация лучшей модели на dev (greedy)

Лучший trial по `harmonic_in_out_cer` загружается из ckpt и оценивается на dev
single-view (без CR-CTC и без `torch.compile`).


In [ ]:
best_result = trial_results_sorted[0]
best_hp = best_result["hp"]
best_ckpt = best_result["best_ckpt_path"]
print(f"Best trial: #{best_result['trial']}, hm_cer={best_result['best_monitored']:.4f}")
print(f"ckpt: {best_ckpt}")

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size,
    d_model=best_hp["d_model"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    augmenter=None,
    target_samplerate=FIXED["samplerate"],
)
dev_loader_eval = DataLoader(
    dev_ds_eval,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=4,
)

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(digits_to_words(t) for t in batch.transcriptions)
        spks.extend(batch.spk_ids)

cer = compute_cer(refs, predictions)
per_spk = compute_per_speaker_cer(refs, predictions, spks)
print(f"Dev CER (greedy): {cer:.4f}")
print(f"Per-speaker CER: {per_spk}")
